# E5 large - RESUME (session 2 of 3)

Epoch 0 finished in 6.98 h and stopped cleanly, exactly as the budget
intends. This picks up at epoch 1 from that checkpoint.

**The previous version's output must be attached as an input.** The guard
in the restore cell is armed, so if the checkpoint is not found this
notebook refuses to run rather than quietly retraining from scratch and
costing another seven hours.


# E5: encoder capacity

The first attempt at this run died with CUDA out of memory, having taken all
but 57 MiB of a 14.56 GiB T4. ModernBERT-large is 395M parameters against
base's 149M, and at sequence length 512 its activations do not fit.

The fix is gradient checkpointing, which recomputes activations during the
backward pass instead of storing them. It is the right lever precisely because
it changes nothing about the result: the gradients and the parameter updates
are what they would have been without it, and only the wall clock moves, by
roughly a third. Reducing the batch further or shortening the sequence would
have been cheaper and would have changed the experiment.

Two deviations from the reference configuration remain, and both belong in the
paper: batch size 16 rather than 32, and checkpointing on. Sequence length
stays at 512 on purpose, because shortening it would mean the large model saw
less of each file than the reference did, and the ablation could then no longer
separate capacity from truncation.

**This needs three sessions.** Each stops cleanly after the epoch that fits in
`--max-hours 10`; re-run with the previous version's output attached.

## What this is for

The reporting rule matters and is easy to get wrong. The finding is **not** that
this setting is better or optimal. It is that the collapse under compound shift
is present in every configuration we tried, so it cannot be dismissed as an
artefact of one particular set of free parameters.

The corpus is built exactly as the original GPU build, and only
`kaggle_e5_large.yaml` differs from `kaggle.yaml`, in one axis.

## Settings

| Setting | Value |
|---|---|
| **Accelerator** | `GPU T4 x2` |
| **Internet** | `On` |
| **Persistence** | `Files only` |

**Save Version -> Save & Run All (Commit).** Budget about 26 h over three sessions. The trainer is
given `--max-hours 10`, so it stops after the last epoch that fits rather than
being killed mid-epoch, and re-running this notebook with the previous
version's output attached continues from that checkpoint.


In [ ]:
import os, sys, time, subprocess, shutil, pathlib, json

T0 = time.time()
def elapsed(label=""):
    m = (time.time() - T0) / 60
    print(f"[{m:6.1f} min] {label}", flush=True)

def run(cmd):
    """Run a stage and stop the notebook if it fails, rather than letting the
    next cell train on whatever stale data is lying around."""
    print(">>", " ".join(str(c) for c in cmd), flush=True)
    r = subprocess.run([sys.executable, "-u", *[str(c) for c in cmd]])
    if r.returncode != 0:
        raise SystemExit(f"FAILED: {' '.join(str(c) for c in cmd)}")

print("Python", sys.version.split()[0])
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU{i}: {p.name}  {p.total_memory/1e9:.1f} GB")
else:
    raise SystemExit("No GPU. Set Accelerator to GPU T4 x2 in the right panel.")

## 1. Code

In [ ]:
WORK = pathlib.Path("/kaggle/working/project")
WORK.mkdir(parents=True, exist_ok=True)

def find_aicd():
    root = pathlib.Path("/kaggle/input")
    if not root.exists():
        return None
    for cand in root.rglob("aicd"):
        if (cand / "config.py").exists() and (cand / "models").is_dir():
            return cand
    return None

src = find_aicd()
if src is None:
    raise SystemExit(
        "aicd/ not found under /kaggle/input.\n"
        "Right panel -> Input -> Add Input -> Datasets, then add the dataset\n"
        "you created from aicd-code.zip.")

dest = WORK / "aicd"
if dest.exists():
    shutil.rmtree(dest)
shutil.copytree(src, dest)
os.chdir(WORK)
sys.path.insert(0, str(WORK))
print("code ->", dest)
if not (dest / "data" / "exposure_arms.py").exists():
    raise SystemExit("This code dataset predates E1. Re-upload aicd-code.zip.")

## 2. Dependencies

In [ ]:
pkgs = ["xgboost", "tree-sitter", "tree-sitter-language-pack",
        "datasets", "shap", "pyyaml", "scikit-learn", "pyarrow", "datasketch"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
import importlib
for m in ["xgboost", "sklearn", "transformers", "datasets", "yaml", "datasketch"]:
    mod = importlib.import_module(m)
    print(f"  ok  {m:14s} {getattr(mod, '__version__', '')}")
elapsed("deps")

## 3. Resume, if a previous run is attached

In [ ]:
# Resume support. A fresh Kaggle session starts with an empty working
# directory, so `--resume` alone finds nothing, prints "starting fresh" and
# silently retrains from epoch 0. That is how seven hours disappear without
# anyone noticing, so the checkpoint is restored explicitly here.
#
# The corpus is restored too, and that matters more than it looks. Rebuilding
# is deterministic given the seed but NOT across library versions: the same
# pipeline produced 417,645 rows earlier and 417,431 today, a parse-filter
# difference. Resuming a checkpoint onto a corpus it was not trained on would
# be quietly wrong, so if a previous split is available we use it and skip the
# rebuild entirely.
#
# Attach the previous version's output: Input -> Add Input -> Notebook Output.
# Set True when this notebook is pushed as a RESUME. A resume that silently
# finds nothing and trains from scratch is the expensive failure: it looks like
# a normal run and costs a full session. With this on, the notebook refuses.
REQUIRE_RESUME = True

import torch
art = WORK / "aicd" / "artifacts"
(art / "data").mkdir(parents=True, exist_ok=True)
CKPT = "branch_a_e5_large_ckpt.pt"

def newest(pattern):
    hits = sorted(pathlib.Path("/kaggle/input").rglob(pattern),
                  key=lambda q: q.stat().st_mtime, reverse=True)
    return hits[0] if hits else None

sp = newest("splits.parquet")
RESTORED_SPLITS = False
if sp is not None:
    shutil.copy(sp, art / "data" / "splits.parquet")
    import pandas as _pd
    _n = len(_pd.read_parquet(art / "data" / "splits.parquet", columns=["label"]))
    print(f"restored splits.parquet from {sp}  ({_n:,} rows)")
    RESTORED_SPLITS = True
else:
    print("no previous splits.parquet found; the corpus will be built fresh")

ck = newest(CKPT)
if ck is None:
    if REQUIRE_RESUME:
        raise SystemExit(
            f"This notebook was pushed as a RESUME but no {CKPT} was found "
            "under /kaggle/input. Training from scratch here would waste a "
            "whole session and look like success. Attach the previous "
            "version's output (Input -> Add Input -> Notebook Output) and "
            "re-run.")
    print(f"no {CKPT} under /kaggle/input -- this will train from epoch 0.")
    print("If you meant to resume, attach the previous version's output.")
else:
    shutil.copy(ck, art / CKPT)
    _c = torch.load(art / CKPT, map_location="cpu", weights_only=False)
    print(f"restored {CKPT} from {ck}")
    print(f"  holds epoch {_c['epoch']}, so training resumes at epoch {_c['epoch'] + 1}")
    if not RESTORED_SPLITS:
        raise SystemExit(
            "A checkpoint was restored but its corpus was not. Rebuilding may "
            "produce a different split than the one this checkpoint was "
            "trained on, which would make the resumed run unsound. Attach the "
            "previous output so splits.parquet comes with it.")


## 3. Build the corpus

One train shard, exactly the original GPU build: 493,850 raw rows filtering to 417,645, of which 196,854 are training.

In [ ]:
CFG = "kaggle.yaml"

# One train shard, not three. One shard plus dev and test is 493,850 raw rows
# which filter to the 417,645 of the original GPU build, with 196,854 of them
# training. Three shards is the matched-scale corpus and gives roughly 545,000
# training rows, which is a different experiment.
if RESTORED_SPLITS:
    print("corpus restored from the previous run; skipping the rebuild so the")
    print("resumed model continues on exactly the data it was trained on.")
    import pandas as pd
    _sp = pd.read_parquet(WORK / "aicd" / "artifacts" / "data" / "splits.parquet",
                          columns=["split"])
    rows = int((_sp["split"] == "train").sum())
    print(f"training rows: {rows:,}")
else:
    run(["-m", "aicd.data.download", "--config", CFG, "--train-shards", "1"])
    elapsed("downloaded 1 shard")

    for stage in ["normalize", "filter", "splits"]:
        run(["-m", f"aicd.data.{stage}", "--config", CFG])
        elapsed(stage)

    run(["-m", "pytest", "aicd/tests/", "-q"])
    elapsed("integrity tests passed")

    sp_report = json.load(open(WORK / "aicd" / "eval" / "reports" / "splits.json"))
    rows = sp_report["train"]["rows"]
    print(f"\ntraining rows: {rows:,}   (expected 196,854)")
    if abs(rows - 196854) > 5000:
        raise SystemExit(
            f"Got {rows:,} training rows, expected about 196,854. This notebook "
            "must reproduce the original GPU build exactly, or the arms are not "
            "comparable with the existing model.")

## 4. Train with encoder capacity

In [ ]:
run(["-m", "aicd.models.modernbert_triplet",
     "--config", "kaggle_e5_large.yaml",
     "--tag", "e5_large", "--max-hours", "10", "--resume"])
elapsed("e5_large trained and evaluated")

## 5. Against the reference configuration

In [ ]:
rep = WORK / "aicd" / "eval" / "reports" / "branch_a_e5_large.json"
r = json.load(open(rep))["slices"]
REFERENCE = {"s1_in_distribution": 0.8977, "s2_unseen_generator": 0.8685,
             "s3_unseen_language": 0.5667, "s4_unseen_domain": 0.4029,
             "s5_compound": 0.2378}
print(f"{'condition':24s} {'this run':>10s} {'reference':>10s} {'delta':>8s}")
print("-" * 56)
for s, ref in REFERENCE.items():
    if s in r:
        v = r[s]["macro_f1"]
        print(f"{s:24s} {v:10.4f} {ref:10.4f} {v-ref:+8.4f}")
s1 = r.get("s1_in_distribution", {}).get("macro_f1")
s5 = r.get("s5_compound", {}).get("macro_f1")
if s1 and s5:
    print(f"\nS1 -> S5: {s1:.4f} -> {s5:.4f}  (drop {s1-s5:.4f})")
    print(f"reference drop:                    {0.8977-0.2378:.4f}")
    if s5 < 0.45:
        print("\nThe collapse is present in this configuration too.")
    else:
        print("\nThe collapse is NOT present here. That is a real finding and")
        print("changes the paper: part of the effect was configuration-specific.")

## 6. Save

In [ ]:
OUT = pathlib.Path("/kaggle/working/results")
OUT.mkdir(parents=True, exist_ok=True)

reports = WORK / "aicd" / "eval" / "reports"
if reports.exists():
    shutil.copytree(reports, OUT / "reports", dirs_exist_ok=True)

art = WORK / "aicd" / "artifacts"
npy = OUT / "arrays"
npy.mkdir(exist_ok=True)
n = 0
for f in art.glob("proba_a*.npy"):
    shutil.copy(f, npy / f.name); n += 1
for name in ("arm_report.json", "splits_arms.parquet"):
    p = art / "data" / name
    if p.exists() and p.stat().st_size < 200e6:
        shutil.copy(p, OUT / name)
masks = art / "data" / "arm_masks"
if masks.exists():
    shutil.copytree(masks, OUT / "arm_masks", dirs_exist_ok=True)
if (art / "kaggle").exists():
    for f in (art / "kaggle").glob("*"):
        if f.is_file() and f.stat().st_size < 200e6:
            shutil.copy(f, npy / f.name); n += 1

shutil.make_archive("/kaggle/working/results", "zip", OUT)
print(f"copied {n} arrays")
print("-> /kaggle/working/results.zip  (Output tab)")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"  {p.stat().st_size/1024:8.0f} KB  {p.relative_to(OUT)}")
elapsed("saved")